# 📘 Tutorial 1: From Gaussian Processes to BoTorch Models

> This notebook is provided in a clean, non-executed state for readability and reproducibility.

In **Part 3**, we built Gaussian Process surrogates from first principles.

We introduced:
- kernels as covariance functions,
- GP priors and posteriors,
- predictive mean and predictive uncertainty,
- and the role of Gaussian Processes in Bayesian Optimisation.

In this tutorial, we take the next practical step:

> **How are those same Gaussian Process ideas represented in BoTorch?**

This notebook is **not yet** about full Bayesian Optimisation.
Instead, its purpose is to show how a Gaussian Process surrogate can be built, fitted, and queried using BoTorch in a way that stays closely connected to the intuition developed earlier.

We will focus mainly on **one-dimensional examples** so that the model behaviour remains easy to interpret visually.
By the end of the notebook, the main message should be clear:

> **BoTorch does not replace the Gaussian Process ideas from Part 3 — it packages them into practical model objects that can be trained and used efficiently.**

---

**This tutorial is designed to shift perspective**
- from *“I can derive and implement a GP posterior by hand”*
- to *“I can now build, fit, and query the same surrogate using BoTorch.”*

---

**The emphasis is on developing intuition for**
- how BoTorch expects training data to be shaped,
- what `SingleTaskGP` represents,
- why input normalisation and output standardisation are included,
- how GP hyperparameters are fit in practice,
- and how the familiar quantities `μ(x)` and `σ(x)` emerge from the BoTorch posterior.

---

**Key ideas explored include**
- one-dimensional regression in BoTorch,
- `SingleTaskGP` as a standard exact GP surrogate,
- `ExactMarginalLogLikelihood` and model fitting,
- posterior mean and posterior variance,
- learned hyperparameters such as lengthscale and noise,
- and fitting the same BoTorch workflow to a more complex one-dimensional function.

---

This tutorial serves as the practical bridge from **Part 3 Gaussian Process intuition** to **Part 4 BoTorch-based Bayesian Optimisation workflows**.

In particular, it prepares the ground for the next tutorial, where the fitted BoTorch GP surrogate will no longer be only a predictive model, but the basis of an **acquisition function** for choosing where to evaluate next.

---

**Recommended prerequisites**
- Completion of **Part 3**
- Basic familiarity with Gaussian Processes, kernels, and predictive uncertainty
- Comfort with PyTorch tensors and one-dimensional plotting

---

**Author**: Angze Li

**Last updated**: 2026-04-05

**Version**: v1.0

## 🔧 Setup

In [ ]:
import torch
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.double)
torch.manual_seed(0)


def style_ax(ax):
    for spine in ax.spines.values():
        spine.set_linewidth(1.8)
    ax.tick_params(axis="both", labelsize=14)
    for t in ax.get_xticklabels() + ax.get_yticklabels():
        t.set_fontweight("bold")

## 1. BoTorch and GPyTorch modules used in this tutorial

Before building the model, we import the core BoTorch and GPyTorch components that will define, fit, and query a Gaussian Process surrogate.

This import block contains the main objects needed for the first practical GP workflow in BoTorch.

---

### What each imported module does

- `SingleTaskGP`
  This is BoTorch’s standard exact Gaussian Process model for a **single scalar output**.
  It is the main surrogate model used in this tutorial.

- `Normalize`
  This input transform rescales the training inputs internally to a standard range.
  In practice, BoTorch models are usually more stable and easier to fit when the inputs are normalised.

- `Standardize`
  This outcome transform standardises the observed outputs internally.
  This improves numerical conditioning and makes GP hyperparameter fitting more reliable.

- `fit_gpytorch_mll`
  This is the fitting routine used to optimise the GP hyperparameters.
  In other words, it is the function that trains the Gaussian Process model after it has been constructed.

- `GPyTorchPosterior`
  This is the posterior object type returned when we query the fitted BoTorch GP model.
  It stores the predictive distribution, including the posterior mean and posterior variance.

- `ExactMarginalLogLikelihood`
  This is the objective used to fit an exact GP model.
  Conceptually, it tells the model to choose hyperparameters that make the observed data as plausible as possible under the Gaussian Process.

---

### Why these imports matter

Together, these modules correspond closely to the GP concepts developed in Part 3:

- `SingleTaskGP` gives the surrogate model
- `Normalize` and `Standardize` handle practical scaling
- `ExactMarginalLogLikelihood` defines the fitting objective
- `fit_gpytorch_mll` performs the actual training
- `GPyTorchPosterior` gives access to the fitted predictive distribution

So this import block is the first point where the Gaussian Process ideas from Part 3 begin to appear as concrete BoTorch objects.

In [ ]:
from botorch.models import SingleTaskGP
from botorch.models.transforms import Normalize, Standardize
from botorch.fit import fit_gpytorch_mll
from botorch.posteriors import GPyTorchPosterior
from gpytorch.mlls import ExactMarginalLogLikelihood

## 2. A simple one-dimensional objective

Before building a Gaussian Process model in BoTorch, we first define a simple **one-dimensional objective** to serve as the unknown function we want to model.

This is **not yet** a Bayesian Optimisation example.
At this stage, the goal is simply to create a controlled setting in which the Gaussian Process posterior can later be visualised clearly and interpreted easily.

---

### What the code does

The function `objective_1d(x)` defines a smooth one-dimensional objective with three components:

- a broad negative Gaussian-shaped dip,
- a small sinusoidal oscillation,
- and a weak quadratic term.

These are combined to produce a function that is simple enough to plot clearly, but still structured enough that sparse observations will **not** determine it perfectly everywhere.

The code then constructs a dense input grid

$$
x_{\text{dense}} \in [-3,\,3],
$$

and evaluates the objective on that grid to obtain `y_dense`.

Finally, it plots the full objective curve.

---

### Why this objective is useful

This function is deliberately chosen to be more interesting than a trivial parabola, but not so complicated that the tutorial becomes visually cluttered.

It has:

- a clear low-value region,
- some mild local variation,
- and enough curvature that the GP posterior will have something meaningful to learn.

That makes it a good teaching example for the first BoTorch notebook.

---

### Why the input is written as `unsqueeze(-1)`

The line
```python
x_dense = torch.linspace(-2.0, 3.0, 500).unsqueeze(-1)
```
 turns a one-dimensional tensor of shape `(500,)` into a two-dimensional tensor of shape `(500, 1)`.

This matters because BoTorch expects inputs in the shape

$$
(n, d),
$$

where:

- **$n$** is the number of input points,
- **$d$** is the input dimension.

Since this is a one-dimensional problem, we have

$$
d = 1,
$$

so the dense evaluation grid must have shape `(500, 1)` rather than just `(500,)`.

This is the first place where the BoTorch input-shape convention becomes visible.

---

### How to interpret the plot

The plotted curve represents the full unknown objective.
In a real optimisation problem, we would not normally know the function everywhere like this.

It is shown here only for teaching purposes, so that later we can compare:

- the **true objective**,
- the **observed training points**,
- and the **fitted BoTorch GP posterior**.

So this plot gives us the hidden function that the model will try to learn from sparse observations.

---

### Key takeaway

This cell defines the one-dimensional objective that will be used throughout the tutorial and evaluates it on a dense grid for reference.

It provides the underlying function that the BoTorch Gaussian Process model will later try to approximate from limited observed data.

In [ ]:
def objective_1d(x):
    return (
        -0.55 * torch.exp(-0.5 * ((x - 1.1) / 0.45) ** 2)
        + 0.10 * torch.sin(3.0 * x)
        + 0.04 * (x - 0.25) ** 2
        + 0.12
    )

x_dense = torch.linspace(-3.0, 3.0, 500).unsqueeze(-1)
y_dense = objective_1d(x_dense)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0)
ax.set_title("The unknown objective we want to model", fontsize=18, fontweight="bold")
ax.set_xlabel("x", fontsize=22, fontweight="bold")
ax.set_ylabel("f(x)", fontsize=22, fontweight="bold")
style_ax(ax)
plt.show()

## 3. Constructing training data for BoTorch

We now create the small observed dataset that the BoTorch Gaussian Process model will be trained on.

This is the first point where the regression problem takes the explicit tensor form expected by BoTorch.

---

### What the code does

The line
```python
train_X = torch.linspace(-1.5, 1.5, 8).unsqueeze(-1)
```
creates 8 input locations across the interval

$$
[-1.5,\;1.5].
$$

The line
```python
train_Y = objective_1d(train_X)
```
 evaluates the objective at those points.

Then we add a small perturbation using

```python
train_Y = train_Y + 0.03 * torch.rand_like(train_Y)
```

Finally, the code:

- prints the shapes of `train_X` and `train_Y`,
- and plots the observations against the true objective.

So this cell gives us the dataset that the Gaussian Process will treat as its training data.

---

### Why the tensor shapes matter

BoTorch expects inputs and outputs in the shapes

$$
\text{train\_X} \in \mathbb{R}^{n \times d},
\qquad
\text{train\_Y} \in \mathbb{R}^{n \times m},
$$

where:

- **$n$** is the number of observations,
- **$d$** is the input dimension,
- **$m$** is the number of outputs.

Here, we have:

- **$n = 8$**
- **$d = 1$**
- **$m = 1$**

so the shapes become:

- `train_X`: `(8, 1)`
- `train_Y`: `(8, 1)`

This is why the code prints the shapes explicitly: it makes the BoTorch data convention concrete before we build the model.

---

### Why noise is added here

A particularly important part of this cell is the line

```python
train_Y = train_Y + 0.03 * torch.rand_like(train_Y)
```

This adds a small amount of random noise to the observed outputs.

This is useful for several reasons.

#### a. Real experimental data are rarely perfectly clean

In real applications, observations are often affected by:

- measurement error,
- finite instrument precision,
- uncontrolled environmental variation,
- or experimental reproducibility limits.

So adding noise here makes the dataset more realistic.

The GP is therefore not being trained on a perfectly clean mathematical curve, but on slightly imperfect observations of that curve.

#### b. It prevents the example from feeling too artificial

If the data lay exactly on the objective with no deviation at all, the problem would feel overly idealised.

The small noise makes it clearer that the surrogate model is trying to infer the underlying structure from imperfect data, which is much closer to the settings where Bayesian Optimisation is actually useful.

#### c. It prepares the reader for GP likelihood modelling

Gaussian Process models do not just fit a mean curve.
They also model uncertainty, including uncertainty arising from noisy observations.

So adding mild noise here helps motivate why the GP likelihood includes a noise term, and why hyperparameters such as the learned observation noise are meaningful later in the tutorial.

#### d. It makes hyperparameter fitting more realistic

Later in the notebook, the model will learn a lengthscale and a noise level from the data.

That becomes more meaningful when the outputs are not perfectly noise-free.

So this small perturbation helps the fitting step behave more like a practical regression problem rather than a purely idealised interpolation exercise.

---

### How to interpret the plot

The scatter points are the observed data that will be given to the BoTorch GP model.

The smooth curve is the true objective, shown only for reference.

This lets us see two things immediately:

- the observations are sparse,
- and they do not lie exactly on the underlying function because of the added noise.

That is exactly the situation a Gaussian Process surrogate is designed to handle.

---

### Key takeaway

This cell constructs the training dataset used for BoTorch GP fitting.

It also adds a small amount of output noise so that the example better reflects realistic regression settings, where observations are imperfect and the surrogate must learn an underlying trend rather than simply interpolate a perfectly clean function.

In [ ]:
train_X = torch.linspace(-1.5, 1.5, 8).unsqueeze(-1)
train_Y = objective_1d(train_X)
train_Y = train_Y + 0.03 * torch.rand_like(train_Y)
print("train_X shape:", tuple(train_X.shape))
print("train_Y shape:", tuple(train_Y.shape))
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="True objective")
ax.scatter(train_X.numpy(), train_Y.numpy(), s=45, zorder=3, label="Observations")
ax.set_title("Observed data for BoTorch GP fitting", fontsize=18, fontweight="bold")
ax.set_xlabel("x", fontsize=22, fontweight="bold")
ax.set_ylabel("f(x)", fontsize=22, fontweight="bold")
ax.legend(prop={"size": 10, "weight": "bold"})
style_ax(ax)
plt.show()

## 4. Building a `SingleTaskGP` model in BoTorch

We now construct the actual Gaussian Process surrogate.

The line `gp = SingleTaskGP(...)` creates a **single-output exact GP model** using the observed training data `train_X` and `train_Y`.

This is the BoTorch object that will play the role of the surrogate model throughout the rest of the tutorial.

---

### What the code is doing

The model is built from:

- `train_X`: the observed input locations
- `train_Y`: the observed output values

and two transforms:

- `input_transform=Normalize(d=1)`
- `outcome_transform=Standardize(m=1)`

So the full statement
```python
gp = SingleTaskGP(
    train_X=train_X,
    train_Y=train_Y,
    input_transform=Normalize(d=1),
    outcome_transform=Standardize(m=1))
```
means:

> build a one-dimensional, single-output Gaussian Process model from the observed data, while internally normalising the inputs and standardising the outputs.

This is a very standard BoTorch modelling choice.

The next cell then checks those transforms explicitly by applying them to the training data and printing the resulting ranges and summary statistics.

---

### Why `SingleTaskGP` is used here

`SingleTaskGP` is BoTorch’s default exact GP model for the case where:

- there is **one objective**
- and each input produces **one scalar output**

That is exactly our setting in this notebook.

So `SingleTaskGP` is the natural BoTorch equivalent of the Gaussian Process surrogate we constructed manually in Part 3.

---

### Why the transforms are included

The two transforms are practical modelling tools that BoTorch applies internally.

#### `Normalize(d=1)`

This transform rescales the one-dimensional inputs to a standard range.

This is useful because GP fitting is usually more stable when inputs are on a comparable numerical scale.
So instead of fitting directly on raw input values, the model works internally with normalised inputs.

In this example, the transform check prints:

- **Original X range:** `-1.5` to `1.5`
- **Normalized X range:** `0.0` to `1.0`

So here, `Normalize(d=1)` is indeed mapping the observed training inputs from the interval `[-1.5, 1.5]` to the unit interval `[0, 1]`.

#### `Standardize(m=1)`

This transform standardises the observed outputs.

That means the model internally works with outputs that have been shifted and rescaled to a more numerically convenient scale.

This helps make hyperparameter fitting more robust and better conditioned.
So although the model is trained from the same data, it is doing so in a more numerically stable internal representation.

Again, the transform check makes this concrete:

- **Original Y mean/std:** `0.006942713470338209`, `0.24782864797899193`
- **Standardized Y mean/std:** `5.551115123125783e-17`, `1.0`

The tiny value `5.551115123125783e-17` is effectively zero up to numerical roundoff, so the transformed outputs are behaving exactly as expected: centered to mean zero and scaled to unit standard deviation.

---

### What the printed output means

When we evaluate `gp`, Python prints a summary of the model structure:

```python
SingleTaskGP(
  (likelihood): GaussianLikelihood(
    (noise_covar): HomoskedasticNoise(
      (noise_prior): LogNormalPrior()
      (raw_noise_constraint): GreaterThan(1.000E-04)
    )
  )
  (mean_module): ConstantMean()
  (covar_module): RBFKernel(
    (lengthscale_prior): LogNormalPrior()
    (raw_lengthscale_constraint): GreaterThan(2.500E-02)
  )
  (outcome_transform): Standardize()
  (input_transform): Normalize()
)
```

This is BoTorch showing the internal components of the GP model.

---

### Interpreting each part of the printed model

#### `likelihood: GaussianLikelihood(...)`

This tells us that the model assumes noisy observations through a Gaussian observation model.

That is exactly what we want in a regression setting where the observed `train_Y` values include small noise.
So this part handles the idea that the data are not perfectly exact evaluations of the underlying function.

#### `noise_covar: HomoskedasticNoise(...)`

This means the model is assuming a **constant observation noise level** across all data points.
In other words, every observation is treated as having the same noise variance.

This is a standard and sensible starting assumption for simple GP regression.

#### `noise_prior: LogNormalPrior()`

This tells us that the observation noise parameter is given a log-normal prior.
You do not need to think too deeply about this for now, but it means the model does not treat the noise level as completely unconstrained.
Instead, it begins with a prior belief over plausible noise values.

#### `raw_noise_constraint: GreaterThan(1.000E-04)`

This is a numerical constraint ensuring that the learned noise remains positive and does not collapse to an invalid or dangerously tiny value.
This helps with stability during fitting.

---

#### `mean_module: ConstantMean()`

This tells us that the GP mean function starts as a constant baseline.
So before conditioning on the data, the prior mean is simply a constant value rather than a more complicated deterministic trend.

This is a common and standard GP choice.

---

#### `covar_module: RBFKernel(...)`

This is the covariance kernel used by the Gaussian Process.

The **RBF kernel** is exactly the same kernel family we studied in Part 3.
So this is one of the most important conceptual bridges in the notebook:

> the BoTorch GP is still using the same GP ingredients — it is just packaging them for us.

The RBF kernel implies smoothness and strong local correlation between nearby inputs.

#### `lengthscale_prior: LogNormalPrior()`

This says that the kernel lengthscale parameter is also given a log-normal prior.
Again, this helps regularise the model and guide fitting toward reasonable parameter values.

#### `raw_lengthscale_constraint: GreaterThan(2.500E-02)`

This enforces a lower bound on the lengthscale parameter.
It prevents the kernel from collapsing to pathologically tiny correlation scales during optimisation.

---

#### `outcome_transform: Standardize()`

This confirms that the outputs are being standardised internally, exactly as we requested.

#### `input_transform: Normalize()`

This confirms that the inputs are being normalised internally, exactly as we requested.

---

### Why this output is useful

The printed model summary is valuable because it makes clear that BoTorch is not just a mysterious black box.

It is explicitly constructing a GP model with:

- a likelihood,
- a mean function,
- a covariance kernel,
- and practical numerical transforms.

The transform check is useful for the same reason: it shows numerically that the model really is doing what the transform names suggest.

So even though the implementation is now library-based, the underlying modelling ingredients are still recognisable.

This is the main pedagogical point of these two cells:

> the Gaussian Process ideas from Part 3 are still present — BoTorch is just organising them into a reusable model object.

---

### Key takeaway

These two cells together construct a BoTorch `SingleTaskGP` surrogate from the training data, display its internal structure, and confirm numerically that the input and output transforms are behaving as expected.

The printed output and transform check together show that the model contains:

- a Gaussian likelihood for noisy observations,
- a constant mean function,
- an RBF kernel,
- and internally normalised inputs and standardised outputs.

So this is the practical BoTorch version of the Gaussian Process surrogate we built conceptually in Part 3.

In [ ]:
gp = SingleTaskGP(
    train_X=train_X,
    train_Y=train_Y,
    input_transform=Normalize(d=1),
    outcome_transform=Standardize(m=1),
)
gp

In [ ]:
X_normalized = gp.input_transform(train_X)
Y_standardized, _ = gp.outcome_transform(train_Y)

print("Original X range:", float(train_X.min()), "to", float(train_X.max()))
print("Normalized X range:", float(X_normalized.min()), "to", float(X_normalized.max()))

print("Original Y mean/std:", float(train_Y.mean()), float(train_Y.std()))
print("Standardized Y mean/std:", float(Y_standardized.mean()), float(Y_standardized.std()))

## 5. Defining and fitting the marginal log likelihood

Now that the Gaussian Process model has been constructed, the next step is to **fit its hyperparameters**.

This is done in two stages:

- first, we define the fitting objective using `ExactMarginalLogLikelihood`
- then, we optimise that objective using `fit_gpytorch_mll(mll)`

---

### What the code does

The line
```python
mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
```
creates the **exact marginal log likelihood** associated with the Gaussian Process model `gp`.

This object combines:

- the GP model itself,
- and its Gaussian likelihood,

into the standard training objective used for exact GP regression.

Then the line
```python
fit_gpytorch_mll(mll)
```
performs the actual optimisation of that objective.

So, in plain language, these two lines mean:

> define the criterion that tells us how well the GP explains the observed data, and then adjust the model’s hyperparameters to make that criterion as good as possible.

---

### What is being fitted?

At this stage, the model already has a structure:

- a constant mean function,
- an RBF kernel,
- a Gaussian likelihood,
- and the input/output transforms.

But several important quantities inside that structure are still not yet tuned to the data.

These include, for example:

- the kernel **lengthscale**
- the kernel **output scale** or covariance strength
- the observation **noise level**
- and any other trainable GP parameters associated with the model

So fitting the model means learning these values from the observed dataset rather than fixing them by hand.

---

### What is the marginal log likelihood?

The **marginal log likelihood** is the standard training objective for exact Gaussian Processes.

Conceptually, it asks:

> under the current GP model and its hyperparameters, how probable are the observed data?

So if the hyperparameters are well chosen, the model should assign relatively high probability to the training observations.

If the hyperparameters are poor, the data will look less plausible under the model.

This is why maximising the marginal log likelihood is a natural way to train a GP.

It balances two things automatically:

- fitting the observed data well,
- while also avoiding unnecessarily complicated function behaviour.

So it is not just trying to interpolate the data as aggressively as possible.
It is trying to find a set of GP hyperparameters that gives a good probabilistic explanation of the observations.

---

### What the printed `mll` object means

When we evaluate `mll`, Python prints the full object summary:

```python
ExactMarginalLogLikelihood(
  (likelihood): GaussianLikelihood(
    (noise_covar): HomoskedasticNoise(
      (noise_prior): LogNormalPrior()
      (raw_noise_constraint): GreaterThan(1.000E-04)
    )
  )
  (model): SingleTaskGP(
    (likelihood): GaussianLikelihood(
      (noise_covar): HomoskedasticNoise(
        (noise_prior): LogNormalPrior()
        (raw_noise_constraint): GreaterThan(1.000E-04)
      )
    )
    (mean_module): ConstantMean()
    (covar_module): RBFKernel(
      (lengthscale_prior): LogNormalPrior()
      (raw_lengthscale_constraint): GreaterThan(2.500E-02)
    )
    (outcome_transform): Standardize()
    (input_transform): Normalize()
  )
)
```

This printed structure makes an important point very explicit:

- the `ExactMarginalLogLikelihood` is built from the **Gaussian likelihood**,
- and it is tied directly to the **`SingleTaskGP` model** we just constructed.

So the fitting objective is not some separate arbitrary loss function.
It is the standard exact GP regression objective associated with this specific model.

The printed output also reminds us that the object being fit still contains the familiar modelling ingredients:

- a **Gaussian likelihood** for noisy observations,
- a **constant mean** prior,
- an **RBF kernel**,
- and the same **input/output transforms** we specified earlier.

So even at the level of the fitting objective, the structure of the Gaussian Process model is still fully visible.

---

### What does `fit_gpytorch_mll(mll)` actually do?

The function `fit_gpytorch_mll(mll)` performs the **numerical optimisation** of the marginal log likelihood.

In practice, this means it:

- looks at all trainable parameters inside the GP model,
- computes how the marginal log likelihood depends on them,
- and iteratively updates those parameters to improve the fit.

So when you run `fit_gpytorch_mll(mll)`, BoTorch / GPyTorch is effectively saying:

> “adjust the GP hyperparameters until this model explains the observed data as well as possible under the exact GP regression objective.”

This is the practical BoTorch equivalent of “training the surrogate model.”


---

### Why this matters

This is an important conceptual shift compared with the earlier first-principles notebooks.

In Part 3, we often chose quantities such as the kernel lengthscale manually in order to study their effect.

Here, we are no longer setting those values by hand.

Instead, BoTorch is **learning them from the data** through the marginal log likelihood optimisation step.

That is one of the main practical advantages of using a mature GP library:

- the model structure is clear,
- but the hyperparameter tuning is handled automatically.

---

### Key takeaway

The line `mll = ExactMarginalLogLikelihood(gp.likelihood, gp)` defines the standard exact GP training objective, and `fit_gpytorch_mll(mll)` then optimises that objective to learn the model hyperparameters from data.

So this is the step where the BoTorch Gaussian Process stops being just a model specification and becomes a **fitted surrogate model**.

In [ ]:
mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
mll
fit_gpytorch_mll(mll)

## 6. Inspecting the learned hyperparameters

After fitting the Gaussian Process, we can inspect two of its most important learned hyperparameters:

- the **kernel lengthscale**
- the **observation noise**

The code extracts these from the fitted BoTorch model and prints them directly.

---

### What the code does

The line `gp.covar_module.lengthscale` accesses the learned **RBF kernel lengthscale**.

This parameter controls how quickly the function is allowed to vary with the input:

- a **large** lengthscale implies a smoother, more slowly varying function
- a **small** lengthscale implies a more rapidly varying function

The line `gp.likelihood.noise` accesses the learned **observation noise level**.

This tells us how much residual variation the model attributes to noisy observations rather than to the underlying latent function itself.

So these two printed numbers give a first numerical summary of what the GP has learned from the data.

---

### Interpreting the output

The fitted model gives:

- **Learned lengthscale:** `0.1675`
- **Learned noise:** `0.005990`

#### Learned lengthscale = `0.1675`

This is a fairly short lengthscale relative to the normalised input domain `[0,1]`.

That means the fitted GP is allowing the latent function to vary on a relatively local scale, rather than forcing it to be extremely smooth across the whole domain.

This is reasonable here, because the objective contains a noticeable dip and some local curvature, so the model needs enough flexibility to represent that structure.

#### Learned noise = `0.005990`

This is the fitted observation noise parameter of the Gaussian likelihood.

It is important to remember that this number is **not directly the same thing** as the `0.03` that appeared in the synthetic data generation step.

---

### Does the learned noise make sense if the code used `0.03`?

Yes — it can still make sense, but we need to be careful about what the `0.03` in the code actually means.

The training data were generated using

`train_Y = train_Y + 0.03 * torch.rand_like(train_Y)`.

That does **not** mean “Gaussian noise with standard deviation `0.03`.”

Instead, it means:

- a random **uniform perturbation**
- with values between `0` and `0.03`

So there are two important consequences.

#### 1. The added noise is not zero-centred

Because `torch.rand_like(train_Y)` produces values in `[0,1)`, the perturbation lies in

$$
[0,\;0.03).
$$

So the noise is always **positive**, not centred around zero.

That means this is not a standard symmetric Gaussian noise model.
The GP likelihood, however, is modelling the residual noise as Gaussian.

So we should not expect the learned likelihood noise to match the data-generation constant `0.03` in a simple one-to-one way.

#### 2. The printed GP noise is a variance-like model parameter, not the same raw scale as the injected perturbation

The value `0.005990` comes from the GP likelihood after the model has also applied an **outcome standardisation transform**.

So the model is learning noise in its internally standardised output space, not directly in the raw original output units.

Because of that, comparing `0.005990` directly to `0.03` is not the right comparison.

---

### Where is the kernel variance?

A natural question here is:

> in Part 3, we always defined both a **lengthscale** and a **variance** in the RBF kernel, so why are we only printing lengthscale and noise here?

The reason is that the current BoTorch model is using a covariance module printed as `RBFKernel(...)`, not a separately scaled kernel such as a `ScaleKernel(RBFKernel(...))`.

So in this model, the explicitly visible learned scalar hyperparameters are:

- the **RBF lengthscale**
- and the **likelihood noise**

but not a separately printed kernel amplitude or kernel variance term.

This does **not** mean variance has disappeared from Gaussian Process modelling altogether.
It just means that, in this particular BoTorch model specification, we are not exposing a separate multiplicative kernel variance parameter in the same way we did in Part 3.

It is also true that the outputs have already been transformed using `Standardize(m=1)`, so the training targets are internally centred to mean zero and scaled to unit standard deviation.

Because of that, adding and interpreting a separate kernel variance parameter becomes less essential in this simple tutorial setting:

- the output scale has already been normalised,
- the model still learns how quickly the function varies through the **lengthscale**,
- and it still learns how noisy the observations are through the **likelihood noise**.

So the short answer is:

> since `train_Y` has already been standardised internally, this tutorial does not need to separately emphasise a kernel variance parameter in order to build a sensible GP surrogate.

That said, this is a modelling choice, not a universal rule.

If we wanted a closer match to the Part 3 kernel form

$$
k(x_i, x_j) = \text{variance} \cdot \exp\!\left(-\frac{(x_i-x_j)^2}{2\ell^2}\right),
$$

then we would use a kernel with an explicit outputscale term, so that both **lengthscale** and **kernel variance** were learned and printed separately.

---

### Key takeaway

This cell shows that, after fitting, the BoTorch GP has learned both:

- a **lengthscale** that controls how locally the function can vary
- and a **noise level** that reflects uncertainty in the observations

The learned values

- `lengthscale = 0.1675`
- `noise = 0.005990`

are plausible for this dataset.

And although the synthetic data were perturbed using `0.03 * torch.rand_like(train_Y)`, that `0.03` should **not** be expected to match the learned GP noise directly, because the injected perturbation is uniform and one-sided, while the GP likelihood noise is learned in a transformed Gaussian-noise model.

In [ ]:
lengthscale = gp.covar_module.lengthscale.detach().cpu().item()
noise = gp.likelihood.noise.detach().cpu().item()

print(f"Learned lengthscale: {lengthscale:.4f}")
print(f"Learned noise:       {noise:.6f}")

## 7. Querying the BoTorch posterior

Now that the Gaussian Process has been fitted, we can ask the model for its **posterior distribution** over a dense set of input points.

This is the BoTorch version of the same step we studied repeatedly in Part 3:
once the GP has been conditioned on the observed data, it produces a predictive distribution at new inputs.

---

### What the code does

The line

```python
gp.eval()
```
switches the model into **evaluation mode**.

Then the line
```python
posterior = gp.posterior(x_dense)
```
asks the fitted GP model for its posterior distribution over the dense input grid `x_dense`.

Finally, the code prints:

- the type of the posterior object
- the shape of the posterior mean
- and the shape of the posterior variance

So this cell is the first place where the fitted BoTorch model is used to produce the familiar predictive quantities of Gaussian Process regression.

---

### What does `gp.eval()` do?

The method `gp.eval()` tells the model that we are no longer in a fitting or training stage.

Instead, we are now using the model for **prediction**.

In practical terms, this means:

- the model stops behaving as something that is currently being updated,
- and begins behaving as a fixed fitted object that can be queried consistently.

This is standard practice in PyTorch-style model workflows.

In this notebook, `gp.eval()` matters because we want the fitted Gaussian Process to use its learned parameters and learned transform state in a stable predictive mode when computing the posterior.

So conceptually, `gp.eval()` means:

> the model has now been trained, and we are ready to use it as a fitted surrogate.

---

### What is `gp.posterior(x_dense)` returning?

The call `gp.posterior(x_dense)` returns the **posterior distribution** of the Gaussian Process evaluated at the test inputs `x_dense`.

This is not just a vector of predicted values.

It is a full probabilistic object that contains the GP’s predictive distribution over those test points.

That is why the printed type is

`<class 'botorch.posteriors.gpytorch.GPyTorchPosterior'>`

So the result is a **posterior object**, not merely a tensor.

This object stores the predictive quantities we care about, including:

- posterior mean
- posterior variance

and, more generally, the predictive Gaussian structure of the fitted model.

---

### Why the shapes are `(500, 1)`

The output shows:

- `posterior mean shape: (500, 1)`
- `posterior variance shape: (500, 1)`

This makes sense because:

- we queried the model at `500` input locations
- and the GP has **one scalar output**

So the predictive mean and predictive variance each have one value per input point, giving shape

$$
(500, 1).
$$

This is exactly consistent with the earlier BoTorch shape convention:

- `n = 500` test points
- `m = 1` output dimension

So the posterior is giving us a one-dimensional predictive mean and one-dimensional predictive variance evaluated across the dense grid.

---

### Why this matters conceptually

This is one of the most important bridge cells in the notebook.

In Part 3, we wrote code by hand to obtain:

- a posterior mean
- a posterior covariance
- and therefore a posterior uncertainty band

Here, BoTorch is giving us the same conceptual objects through a model method.

So this cell makes the key bridge explicit:

> the fitted BoTorch `SingleTaskGP` can now be queried to produce the same GP posterior quantities we previously constructed manually.

This is exactly why BoTorch is useful:
it preserves the Gaussian Process concepts, but packages them into a practical, reusable interface.

---

### Key takeaway

The call `gp.eval()` switches the fitted model into prediction mode, and `gp.posterior(x_dense)` returns the Gaussian Process posterior over the dense input grid.

The printed output confirms that the posterior is a BoTorch posterior object, and that both the posterior mean and posterior variance are returned with shape `(500, 1)`, corresponding to 500 test inputs and 1 output dimension.

In [ ]:
gp.eval()
posterior = gp.posterior(x_dense)

print(type(posterior))
print("posterior mean shape:", tuple(posterior.mean.shape))
print("posterior variance shape:", tuple(posterior.variance.shape))

## 8. Extracting `μ(x)` and `σ(x)` from the BoTorch posterior

The posterior object returned by `gp.posterior(x_dense)` contains the full predictive distribution of the fitted Gaussian Process.

To plot the surrogate in the familiar form used throughout Part 3, we now extract three key quantities from that posterior:

- the **posterior mean**
- the **posterior variance**
- and the **posterior standard deviation**

These correspond directly to the Gaussian Process quantities

$$
\mu(x), \qquad \sigma^2(x), \qquad \sigma(x).
$$

So this short block of code is doing an important conceptual translation:

- `posterior.mean` gives the predictive mean
- `posterior.variance` gives the predictive variance
- `torch.sqrt(...)` converts that variance into the predictive standard deviation

This is the point where the BoTorch posterior object is turned into the familiar GP summary quantities that we can plot and interpret.

---

### Why `.detach()` is used

The calls to `.detach()` remove these tensors from PyTorch’s gradient-tracking system.

That is appropriate here because we are no longer fitting the model.
We are simply extracting fitted predictive quantities for inspection and plotting.

So `.detach()` tells PyTorch:

> these values are now being used as fixed outputs of the fitted model, not as quantities that need further gradient-based optimisation.

---

### Why `clamp_min(1e-12)` is used

The line `posterior.variance.detach().clamp_min(1e-12)` applies a small numerical safeguard before taking the square root.

In theory, a variance should never be negative.

But in practice, due to floating-point roundoff, a predictive variance can occasionally become extremely slightly negative or extremely close to zero in a way that is numerically inconvenient.

So `clamp_min(1e-12)` ensures that the variance is at least a tiny positive number before we compute

$$
\sigma(x) = \sqrt{\sigma^2(x)}.
$$

This avoids numerical problems and makes the code more robust.

---

### Why this step matters

It may look like a small preparation step before plotting, but it is actually one of the most important bridge points in the notebook.

In Part 3, we manually built and interpreted:

- posterior mean
- posterior variance
- uncertainty bands

Here, BoTorch gives us those same quantities through the posterior object.

So this step makes the connection explicit:

> the fitted BoTorch GP is now producing the same predictive quantities we previously constructed by hand.

---

### Key takeaway

This block extracts the fitted GP’s posterior mean, variance, and standard deviation from the BoTorch posterior object.

That gives us the familiar quantities `μ(x)` and `σ(x)` needed to plot the Gaussian Process surrogate and its uncertainty band in the same style as Part 3.

In [ ]:
mu_botorch = posterior.mean.detach()
var_botorch = posterior.variance.detach().clamp_min(1e-12)
sigma_botorch = torch.sqrt(var_botorch)

## 9. Visualising the fitted BoTorch GP posterior

We are now ready to plot the fitted Gaussian Process surrogate in the familiar form used throughout Part 3.

This figure combines:

- the **true objective**,
- the **observed training points**,
- the **BoTorch posterior mean**,
- and a **posterior uncertainty band**.

So this is the first full visual comparison between:

- the underlying function,
- the sparse observed data,
- and the fitted BoTorch Gaussian Process model.

---

### What the code does

The plot contains four main elements.

#### 1. The true objective

The line `ax.plot(x_dense.numpy(), y_dense.numpy(), ...)` draws the full underlying objective curve.

This is shown only for reference, so that we can compare the surrogate against the actual function.

In a real optimisation setting, we would not normally know this everywhere.

#### 2. The BoTorch posterior mean

The line `ax.plot(x_dense.numpy(), mu_botorch.numpy(), ..., linestyle="--")` plots the posterior mean of the fitted GP.

This is the model’s best estimate of the latent function after conditioning on the observed training data.

So this dashed curve is the BoTorch surrogate itself.

#### 3. The uncertainty band

The `fill_between(...)` block shades the region

$$
\mu(x) \pm 2\sigma(x),
$$

where:

- `mu_botorch` is the posterior mean,
- `sigma_botorch` is the posterior standard deviation.

This gives a visual uncertainty band around the GP mean.

Just as in Part 3, the band is narrower where the model is well supported by observations and wider where the model remains more uncertain.

#### 4. The observed data points

The scatter plot of `train_X` and `train_Y` shows the actual training observations used to fit the model.

These are the points the GP has seen directly.

So the surrogate is being judged relative to a sparse and noisy dataset, not to the true function everywhere.

---

### Why this figure matters

This is one of the most important bridge figures in the whole notebook.

In Part 3, we manually constructed and plotted:

- the posterior mean,
- the posterior uncertainty,
- and the effect of sparse observations on the surrogate.

Here, the same visual story is now being produced through BoTorch.

So this figure makes the key practical connection explicit:

> the BoTorch `SingleTaskGP` is now functioning exactly as a Gaussian Process surrogate should, producing both a predictive mean and a principled uncertainty estimate.

That is the central point of Tutorial 1.

---

### How to interpret the figure

The **dashed GP mean** should be read as the model’s best estimate of the latent function.

The **shaded band** shows where the model is more or less certain.

The **observations** anchor the surrogate locally, while the GP interpolates and extrapolates probabilistically between and beyond them.

So when you look at the plot, the main questions are:

- does the GP mean follow the overall structure of the objective?
- is the uncertainty smaller near observed points?
- does the uncertainty grow in regions with weaker data support?

Those are exactly the same questions we asked in Part 3, which is why this figure is such an important bridge into BoTorch.

---

### Key takeaway

This figure shows the fitted BoTorch Gaussian Process in its most interpretable form:

- the posterior mean gives the surrogate prediction,
- the uncertainty band shows where the model is more or less confident,
- and the observations reveal how the surrogate is anchored by data.

So this is the practical BoTorch version of the Gaussian Process posterior we developed conceptually in Part 3.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="True objective")
ax.plot(x_dense.numpy(), mu_botorch.numpy(), linewidth=2.0, linestyle="--", label="BoTorch GP mean")
ax.fill_between(
    x_dense.squeeze(-1).numpy(),
    (mu_botorch - 2.0 * sigma_botorch).squeeze(-1).numpy(),
    (mu_botorch + 2.0 * sigma_botorch).squeeze(-1).numpy(),
    alpha=0.2,
    label=r"$\mu(x)\pm 2\sigma(x)$",
)
ax.scatter(train_X.numpy(), train_Y.numpy(), s=45, zorder=3, label="Observations")
ax.set_title("BoTorch GP posterior", fontsize=18, fontweight="bold")
ax.set_xlabel("x", fontsize=22, fontweight="bold")
ax.set_ylabel("f(x)", fontsize=22, fontweight="bold")
ax.legend(prop={"size": 10, "weight": "bold"}, loc="upper left")
style_ax(ax)
plt.show()

## 10. Inspecting the posterior uncertainty directly

So far, the Gaussian Process posterior has been visualised through the shaded band

$$
\mu(x) \pm 2\sigma(x).
$$

That is often the most intuitive way to see prediction and uncertainty together.

But it is also useful to look at the **posterior standard deviation** on its own.

This isolates the uncertainty structure of the fitted model and makes it easier to see exactly where the Gaussian Process is confident and where it remains uncertain.

---

### What the code does

The code plots

$$
\sigma(x),
$$

the posterior standard deviation of the fitted BoTorch Gaussian Process, across the dense input grid `x_dense`.

So unlike the previous figure, this plot does **not** show:

- the true objective,
- the GP mean,
- or the observations.

It shows only the predictive uncertainty of the model itself.

That makes it a cleaner uncertainty diagnostic.

---

### Why this is useful

When uncertainty is shown only as a shaded band around the mean, it is visually tied to the predicted function values.

That is useful for interpretation, but it can sometimes make the uncertainty structure harder to read directly.

Plotting `σ(x)` by itself helps answer questions like:

- where is the GP most certain?
- where is it most uncertain?
- how strongly does uncertainty depend on the locations of observed data?

So this figure separates the uncertainty story from the mean-prediction story.

---

### How to interpret the curve

The quantity

$$
\sigma(x)
$$

measures the predictive standard deviation of the Gaussian Process at each input location.

A **smaller** value of `σ(x)` means:

- the model is more confident there,
- usually because that region is well supported by nearby observations.

A **larger** value of `σ(x)` means:

- the model is less confident there,
- usually because that region is farther from the observed training data.

So the shape of this curve should reflect the geometry of the observation set.

In particular, we expect uncertainty to be reduced near the training points and larger in regions where data support is weaker.

---

### Why this matters for Bayesian Optimisation

This plot is especially important because, in Bayesian Optimisation, uncertainty is not just a visual feature.

It is part of the decision-making mechanism.

Later in Part 4, acquisition functions such as:

- Expected Improvement,
- Probability of Improvement,
- and Upper/Lower Confidence Bound

will all depend on the GP’s predictive uncertainty.

So understanding the shape of `σ(x)` directly is an important step toward understanding how the surrogate will later guide where to evaluate next.

---

### Connection to Part 3

This is the same conceptual object we studied earlier in the first-principles GP notebooks.

The important difference is that here it is being obtained from the fitted BoTorch posterior rather than from hand-written GP code.

So this figure reinforces the main bridge of Tutorial 1:

> the Gaussian Process uncertainty from Part 3 is still present in exactly the same conceptual form — BoTorch is simply computing it for us through the posterior object.

---

### Key takeaway

This plot shows the fitted GP’s posterior standard deviation

$$
\sigma(x),
$$

directly across the domain.

It isolates the model’s uncertainty from the posterior mean and makes it easier to see how confidence varies with data support — which is crucial both for interpreting Gaussian Process surrogates and for understanding how Bayesian Optimisation will later use them.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_dense.numpy(), sigma_botorch.numpy(), linewidth=2.5)
ax.set_title("Posterior standard deviation", fontsize=18, fontweight="bold")
ax.set_xlabel("x", fontsize=22, fontweight="bold")
ax.set_ylabel(r"$\sigma(x)$", fontsize=22, fontweight="bold")
style_ax(ax)
plt.show()

## 11. A more complex one-dimensional function: the same BoTorch workflow

Up to this point, the tutorial has focused on a relatively simple one-dimensional objective so that the BoTorch Gaussian Process workflow remains easy to interpret.

To close the notebook, we now apply **exactly the same modelling pipeline** to a more structured function.

The purpose is **not yet** to perform Bayesian Optimisation itself.
Instead, it is to show that the BoTorch GP workflow we have just learned —

- define training data,
- build a `SingleTaskGP`,
- fit with `ExactMarginalLogLikelihood`,
- switch to evaluation mode,
- query the posterior,
- and extract `μ(x)` and `σ(x)` —

already works naturally for functions that are much less visually trivial.

---

### What the code does

The code first defines a more complex one-dimensional objective, `objective_1d_complex(x)`.

Compared with the earlier example, this function contains:

- multiple negative Gaussian-shaped dips,
- oscillatory structure at more than one frequency,
- a mild quadratic trend,
- and a tangent-based deformation that makes the overall shape less symmetric and less smooth-looking at first glance.

So this is a function with **multiple local structures** and a richer global shape.

The code then:

- constructs a dense evaluation grid `x_dense_complex`,
- evaluates the true function to obtain `y_dense_complex`,
- defines a set of observed training points `train_X_complex`,
- generates noisy observations `train_Y_complex`,
- builds a BoTorch `SingleTaskGP`,
- fits it using `ExactMarginalLogLikelihood`,
- queries the posterior on the dense grid,
- and extracts the posterior mean and uncertainty.

Finally, the code creates a two-panel figure.

---

### Why this example is useful

This new example is useful because it shows that the same pipeline does **not** depend on the function being especially easy.

Even when the objective has:

- several local low-value regions,
- fine-scale oscillations,
- and more complicated curvature,

the workflow is still the same:

- specify observations,
- fit a GP surrogate,
- and recover the posterior mean and uncertainty band.

So this cell reinforces an important practical point:

> once the modelling pipeline is in place, BoTorch can be applied to functions that are much richer than the first toy example without changing the overall structure of the code.

---

### What the left subplot shows

The left panel displays:

- the **true complex objective**
- and the **observed noisy training points**

This panel answers the question:

> what kind of function is the model being asked to learn, and what information does it actually get to see?

Because the observations are still sparse relative to the structure of the function, this is a genuinely nontrivial regression problem.

That is exactly why it is useful here.

---

### What the right subplot shows

The right panel adds the fitted BoTorch Gaussian Process posterior:

- the **true objective**
- the **posterior mean**
- the **uncertainty band**
  $$
  \mu(x) \pm 2\sigma(x)
  $$
- and the **observations**

This panel answers the question:

> after fitting the BoTorch GP, what does the surrogate believe about this more complex objective?

So the right subplot is the practical payoff of the full modelling pipeline.

---

### Why this is a good closing example for Tutorial 1

This example is a strong way to end the notebook because it shows that Tutorial 1 has already delivered something substantial.

By this point, we are no longer just fitting a GP to an especially gentle toy curve.

We are using the same BoTorch workflow on a function with:

- multiple competing structures,
- local irregularity,
- and a more challenging shape for sparse regression.

That makes the result feel much closer to the kinds of surrogate modelling problems where Bayesian Optimisation becomes genuinely useful.

---

### What the reader should notice

When reading the figure, the main questions are:

- does the GP mean capture the major low-value regions of the function?
- where does the uncertainty remain relatively large?
- how does sparse data support affect the model’s confidence on a more structured objective?

The point is **not** that the GP must recover every fine oscillation perfectly from a small number of observations.

The point is that the BoTorch surrogate is already behaving like a meaningful probabilistic model:

- it tracks the broad structure where the data are informative,
- and it expresses uncertainty where the function is harder to infer.

That is exactly what we need from a surrogate before moving on to acquisition functions.

---

### Key takeaway

This final example shows that the BoTorch GP workflow introduced in Tutorial 1 is already powerful enough to handle a much more structured one-dimensional objective.

So before any acquisition function is introduced, the core practical machinery is already in place:

- define a regression problem,
- build a BoTorch Gaussian Process surrogate,
- fit it to data,
- and recover a posterior mean and uncertainty band over the domain.

In the next tutorial, we will build directly on this fitted-surrogate workflow and ask the next Bayesian Optimisation question:

> **given a BoTorch GP surrogate, where should we evaluate next?**

In [ ]:
def objective_1d_complex(x):
    return (
        -0.48 * torch.exp(-0.5 * ((x + 1.9) / 0.55) ** 2)
        -0.72 * torch.exp(-0.5 * ((x + 0.35) / 0.22) ** 2)
        -1.05 * torch.exp(-0.5 * ((x - 1.75) / 0.30) ** 2)
        +0.10 * torch.sin(2.8 * x)
        +0.06 * torch.sin(6.5 * x + 0.4)
        +0.035 * torch.cos(11.0 * x)
        +0.32 * torch.tan(0.3 * x)
        +0.025 * (x - 0.4) ** 2
        +0.16
    )

x_dense_complex = torch.linspace(-3.0, 3.0, 600).unsqueeze(-1)
y_dense_complex = objective_1d_complex(x_dense_complex)

train_X_complex = torch.linspace(-2, 2, 12).unsqueeze(-1)
train_Y_complex = objective_1d_complex(train_X_complex)
train_Y_complex = train_Y_complex + 0.03 * torch.rand_like(train_Y_complex)


gp_complex = SingleTaskGP(
    train_X=train_X_complex,
    train_Y=train_Y_complex,
    input_transform=Normalize(d=1),
    outcome_transform=Standardize(m=1),
)

mll_complex = ExactMarginalLogLikelihood(gp_complex.likelihood, gp_complex)
fit_gpytorch_mll(mll_complex)

gp_complex.eval()
posterior_complex = gp_complex.posterior(x_dense_complex)

mu_complex = posterior_complex.mean.detach()
var_complex = posterior_complex.variance.detach().clamp_min(1e-12)
sigma_complex = torch.sqrt(var_complex)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

axes[0].plot(x_dense_complex.numpy(), y_dense_complex.numpy(), linewidth=2.0, label="True objective")
axes[0].scatter(train_X_complex.numpy(), train_Y_complex.numpy(), s=45, zorder=3, label="Observations")
axes[0].set_title("A more complex one-dimensional objective", fontsize=18, fontweight="bold")
axes[0].set_xlabel("x", fontsize=22, fontweight="bold")
axes[0].set_ylabel("f(x)", fontsize=22, fontweight="bold")
axes[0].legend(prop={"size": 10, "weight": "bold"}, loc="upper left")
style_ax(axes[0])

axes[1].plot(x_dense_complex.numpy(), y_dense_complex.numpy(), linewidth=2.0, label="True objective")
axes[1].plot(x_dense_complex.numpy(), mu_complex.numpy(), linewidth=2.0, linestyle="--", label="BoTorch GP mean")
axes[1].fill_between(
    x_dense_complex.squeeze(-1).numpy(),
    (mu_complex - 2.0 * sigma_complex).squeeze(-1).numpy(),
    (mu_complex + 2.0 * sigma_complex).squeeze(-1).numpy(),
    alpha=0.2,
    label=r"$\mu(x)\pm 2\sigma(x)$",
)
axes[1].scatter(train_X_complex.numpy(), train_Y_complex.numpy(), s=45, zorder=3, label="Observations")
axes[1].set_title("BoTorch GP fit on a more complex objective", fontsize=18, fontweight="bold")
axes[1].set_xlabel("x", fontsize=22, fontweight="bold")
axes[1].set_ylabel("f(x)", fontsize=22, fontweight="bold")
axes[1].legend(prop={"size": 10, "weight": "bold"}, loc="upper left")
style_ax(axes[1])

plt.tight_layout()
plt.show()

## 🧭 Closing Remarks

In this tutorial, we translated the Gaussian Process ideas of Part 3 into a practical BoTorch workflow.

The central achievement of the notebook is that we can now:

- construct training data in the format BoTorch expects,
- build a `SingleTaskGP` surrogate,
- understand the role of input normalisation and output standardisation,
- fit the model by maximising the exact marginal log likelihood,
- and query the fitted posterior to recover the familiar quantities `μ(x)` and `σ(x)`.

That is the key bridge from theory to implementation.

Earlier in the repository, Gaussian Processes were developed mainly as mathematical objects:
- kernels defined similarity,
- priors expressed assumptions over functions,
- and conditioning on data produced posterior mean and uncertainty.

Here, those same ideas have become concrete BoTorch objects:
- the surrogate is now a `SingleTaskGP`,
- fitting is handled through `ExactMarginalLogLikelihood` and `fit_gpytorch_mll`,
- and the posterior is returned directly as a BoTorch posterior object.

So the conceptual content has not changed.
What has changed is the **level of implementation**.

The final example with a more complex one-dimensional function is important for the same reason.
It shows that the workflow introduced here is already strong enough to handle objectives that are not visually trivial.
Even before any acquisition function is introduced, the core surrogate-modelling machinery is already in place:

- define observed data,
- fit a Gaussian Process surrogate,
- and recover a posterior mean and uncertainty band over the domain.

That is exactly the foundation needed for Bayesian Optimisation.

So the main takeaway from Tutorial 1 is:

> **BoTorch gives us a practical interface for building and fitting Gaussian Process surrogates, while preserving the same predictive mean and uncertainty concepts developed in Part 3.**

In the next tutorial, we will build directly on this fitted-surrogate workflow and ask the next Bayesian Optimisation question:

> **given a BoTorch GP surrogate, where should we evaluate next?**